In [3]:
# src/wp1_baselines_and_significance.py
"""
WP1 Reviewer Fix Pack (with progress indicator)

Adds:
- Seasonal naïve baselines (y_{t-48}, y_{t-336}) treated as degenerate quantile forecasts
  *IMPORTANT*: We evaluate seasonal naïve baselines ONLY on test timestamps where the lagged
  observation exists, otherwise metrics become undefined.
- Historical quantile baseline by time-of-week slot (weekday × half-hour)
- Per-school metrics for pooled and per-school quantile GBMs
- Paired bootstrap across schools (95% CI + win fraction) for pooled vs per-school

Progress:
- Prints progress per school with elapsed time + ETA.

Outputs:
- data/wp1_models_and_baselines_overall.csv
- data/wp1_models_and_baselines_per_school.csv
- data/wp1_pooled_vs_local_bootstrap.csv
"""

import os
import time
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor

TARGET = "y"
QUANTILES = [0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95]
TRAIN_FRAC = 0.8

# 30-min settings
STEPS_PER_DAY = 48
LAG_DAY = 48
LAG_WEEK = 336

# Feature settings (match WP1)
SHORT_LAGS = (1, 2)
SEASONAL_LAGS = (LAG_DAY, LAG_WEEK)
ROLL_WINDOWS = (LAG_DAY, LAG_WEEK)

# Drop to avoid one-hot explosion
DROP_ALWAYS = {"date"}  # if present

# Model hyperparams (keep consistent with your benchmark)
GB_PARAMS = dict(
    max_depth=3,
    n_estimators=300,
    learning_rate=0.05,
    subsample=0.8,
    random_state=42,
)

OUT_OVERALL = "data/wp1_models_and_baselines_overall.csv"
OUT_PER_SCHOOL = "data/wp1_models_and_baselines_per_school.csv"
OUT_BOOT = "data/wp1_pooled_vs_local_bootstrap.csv"


# ---------------- Metrics ----------------
def pinball_loss(y_true, y_pred, q):
    e = y_true - y_pred
    return float(np.mean(np.maximum(q * e, (q - 1) * e)))

def interval_coverage(y_true, lo, hi):
    y_true = np.asarray(y_true)
    return float(np.mean((y_true >= lo) & (y_true <= hi)))

def mean_width(lo, hi):
    return float(np.mean(np.asarray(hi) - np.asarray(lo)))

def mean_pinball(y_true, preds: dict):
    return float(np.mean([pinball_loss(y_true, preds[q], q) for q in QUANTILES]))

def summarise_prob(y_true, preds: dict):
    y_true = np.asarray(y_true, float)
    out = {}
    out["mae_p50"] = float(np.mean(np.abs(y_true - preds[0.50])))
    out["pinball_mean"] = mean_pinball(y_true, preds)
    out["cov50_p25_p75"] = interval_coverage(y_true, preds[0.25], preds[0.75])
    out["cov80_p10_p90"] = interval_coverage(y_true, preds[0.10], preds[0.90])
    out["cov90_p05_p95"] = interval_coverage(y_true, preds[0.05], preds[0.95])
    out["width80_p10_p90"] = mean_width(preds[0.10], preds[0.90])
    return out


# ---------------- Features ----------------
def add_lag_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values(["school_id", "timestamp"]).copy()
    g = df.groupby("school_id", sort=False)[TARGET]

    for lag in list(SHORT_LAGS) + list(SEASONAL_LAGS):
        df[f"y_lag_{lag}"] = g.shift(lag)

    for win in ROLL_WINDOWS:
        # shift(1) so the rolling mean only uses past information
        df[f"y_rollmean_{win}"] = (
            g.shift(1).rolling(win).mean().reset_index(level=0, drop=True)
        )

    return df

def drop_feature_traps(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for c in DROP_ALWAYS:
        if c in df.columns:
            df = df.drop(columns=[c])
    return df

def make_features_fit(df: pd.DataFrame):
    df = drop_feature_traps(df)
    drop = {"timestamp", TARGET}
    X = df[[c for c in df.columns if c not in drop]].copy()

    cat_cols = [c for c in X.columns if str(X[c].dtype) in ("object", "category", "bool")]
    X = pd.get_dummies(X, columns=cat_cols, drop_first=False)

    X = X.replace([np.inf, -np.inf], np.nan).fillna(0)
    return X, X.columns

def make_features_apply(df: pd.DataFrame, fitted_cols):
    df = drop_feature_traps(df)
    drop = {"timestamp", TARGET}
    X = df[[c for c in df.columns if c not in drop]].copy()

    cat_cols = [c for c in X.columns if str(X[c].dtype) in ("object", "category", "bool")]
    X = pd.get_dummies(X, columns=cat_cols, drop_first=False)

    X = X.replace([np.inf, -np.inf], np.nan).fillna(0)
    X = X.reindex(columns=fitted_cols, fill_value=0)
    return X


# ---------------- Models ----------------
def fit_quantile_models(X_train, y_train):
    models = {}
    for q in QUANTILES:
        m = GradientBoostingRegressor(loss="quantile", alpha=q, **GB_PARAMS)
        m.fit(X_train, y_train)
        models[q] = m
    return models

def predict_quantiles(models, X):
    return {q: models[q].predict(X) for q in QUANTILES}


# ---------------- Splits ----------------
def split_by_school(df: pd.DataFrame, train_frac=TRAIN_FRAC):
    parts_train, parts_test = [], []
    for sid, g in df.groupby("school_id", sort=False):
        g = g.sort_values("timestamp")
        cut = int(len(g) * train_frac)
        parts_train.append(g.iloc[:cut])
        parts_test.append(g.iloc[cut:])
    return pd.concat(parts_train, ignore_index=True), pd.concat(parts_test, ignore_index=True)


# ---------------- Baselines ----------------
def seasonal_naive_with_mask(df_train: pd.DataFrame, df_test: pd.DataFrame, lag_steps: int):
    """
    Seasonal naïve per school:
      yhat(t) = y(t - lag_steps)

    Returns:
      yhat (np.ndarray), mask_valid (np.ndarray bool) where yhat is not NaN.
    """
    ref = df_train[["school_id", "timestamp", TARGET]].copy()
    ref["timestamp"] = ref["timestamp"] + pd.Timedelta(minutes=30 * lag_steps)
    ref = ref.rename(columns={TARGET: "yhat"})

    merged = df_test[["school_id", "timestamp"]].merge(ref, how="left", on=["school_id", "timestamp"])
    yhat = merged["yhat"].values
    mask = ~pd.isna(yhat)
    return yhat, mask

def degenerate_quantiles_from_point(yhat_point: np.ndarray):
    return {q: yhat_point.copy() for q in QUANTILES}

def historical_quantile_baseline(df_train: pd.DataFrame, df_test: pd.DataFrame):
    """
    Historical quantiles per school and time-of-week slot.
    Slot = weekday*48 + halfhour_index
    """
    tr = df_train.copy()
    te = df_test.copy()

    def add_slot(d):
        ts = pd.to_datetime(d["timestamp"])
        slot = ts.dt.dayofweek * STEPS_PER_DAY + (ts.dt.hour * 2 + (ts.dt.minute // 30))
        d = d.copy()
        d["slot"] = slot.astype(int)
        return d

    tr = add_slot(tr)
    te = add_slot(te)

    qtab = tr.groupby(["school_id", "slot"])[TARGET].quantile(QUANTILES).unstack(level=-1)
    q_fallback = tr.groupby("school_id")[TARGET].quantile(QUANTILES).unstack(level=-1)

    preds = {}
    keys = list(zip(te["school_id"], te["slot"]))
    for q in QUANTILES:
        vals = []
        for sid, slot in keys:
            try:
                vals.append(qtab.loc[(sid, slot), q])
            except KeyError:
                vals.append(q_fallback.loc[sid, q])
        preds[q] = np.asarray(vals, float)

    return preds


# ---------------- Bootstrap ----------------
def paired_bootstrap(diffs: np.ndarray, n_boot=10000, seed=42):
    rng = np.random.default_rng(seed)
    diffs = np.asarray(diffs, float)
    n = len(diffs)
    boots = np.empty(n_boot, float)
    for b in range(n_boot):
        idx = rng.integers(0, n, size=n)
        boots[b] = np.mean(diffs[idx])
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return float(np.mean(diffs)), float(lo), float(hi)


# ---------------- Progress helper ----------------
def fmt_seconds(s):
    s = max(0, int(s))
    if s < 60:
        return f"{s}s"
    m, s = divmod(s, 60)
    if m < 60:
        return f"{m}m{s:02d}s"
    h, m = divmod(m, 60)
    return f"{h}h{m:02d}m"


# ---------------- Main ----------------
def main(panel_csv="data/schools_panel.csv"):
    os.makedirs("data", exist_ok=True)

    print(f"[WP1-FIX] Loading: {panel_csv}", flush=True)
    df = pd.read_csv(panel_csv, parse_dates=["timestamp"]).copy()
    if "school_id" not in df.columns:
        raise RuntimeError("Expected 'school_id' in panel CSV.")

    df = df.sort_values(["school_id", "timestamp"])
    df = add_lag_features(df)

    # Warm-up drop (lags + target must exist)
    lag_cols = [c for c in df.columns if c.startswith("y_lag_") or c.startswith("y_rollmean_")]
    df = df.dropna(subset=lag_cols + [TARGET]).copy()

    train_df, test_df = split_by_school(df, TRAIN_FRAC)
    print(f"[WP1-FIX] Rows train: {len(train_df):,} | test: {len(test_df):,}", flush=True)

    overall_rows = []
    per_school_rows = []

    # ---- Overall baselines (evaluate ONLY on valid timestamps for seasonal naïve) ----
    for name, lag in [("naive_day", LAG_DAY), ("naive_week", LAG_WEEK)]:
        yhat, mask = seasonal_naive_with_mask(train_df, test_df, lag_steps=lag)
        y_true = test_df[TARGET].values[mask]
        yhat = yhat[mask]
        preds = degenerate_quantiles_from_point(yhat)
        rep = summarise_prob(y_true, preds)
        rep.update({"method": name, "level": "overall", "n_eval": int(mask.sum())})
        overall_rows.append(rep)

    preds_hist = historical_quantile_baseline(train_df, test_df)
    rep_hist = summarise_prob(test_df[TARGET].values, preds_hist)
    rep_hist.update({"method": "hist_quantile_slot", "level": "overall", "n_eval": len(test_df)})
    overall_rows.append(rep_hist)

    # ---- Pooled model ----
    print("[WP1-FIX] Training pooled model (7 quantiles)...", flush=True)
    t_pool = time.time()
    X_tr_pool, cols_pool = make_features_fit(train_df)
    X_te_pool = make_features_apply(test_df, cols_pool)
    y_tr_pool = train_df[TARGET].values
    y_te_pool = test_df[TARGET].values

    pooled_models = fit_quantile_models(X_tr_pool, y_tr_pool)
    pooled_preds_all = predict_quantiles(pooled_models, X_te_pool)

    pooled_overall = summarise_prob(y_te_pool, pooled_preds_all)
    pooled_overall.update({"method": "pooled_gb_quantile", "level": "overall", "n_eval": len(test_df)})
    overall_rows.append(pooled_overall)
    print(f"[WP1-FIX] Pooled done in {fmt_seconds(time.time() - t_pool)}", flush=True)

    # ---- Per-school models + per-school pooled metrics + per-school baselines ----
    schools = list(test_df["school_id"].unique())
    n_schools = len(schools)
    t0 = time.time()

    print(f"[WP1-FIX] Training per-school models ({n_schools} schools × {len(QUANTILES)} quantiles)...", flush=True)

    local_reports = []
    pooled_reports = []

    for k, sid in enumerate(schools, start=1):
        t_school = time.time()
        g_train = train_df[train_df["school_id"] == sid].copy()
        g_test = test_df[test_df["school_id"] == sid].copy()

        # Per-school GB quantile
        X_tr_loc, cols_loc = make_features_fit(g_train.drop(columns=["school_id"]))
        X_te_loc = make_features_apply(g_test.drop(columns=["school_id"]), cols_loc)

        y_tr = g_train[TARGET].values
        y_te = g_test[TARGET].values

        loc_models = fit_quantile_models(X_tr_loc, y_tr)
        loc_preds = predict_quantiles(loc_models, X_te_loc)

        loc_rep = summarise_prob(y_te, loc_preds)
        loc_rep.update({"method": "per_school_gb_quantile", "level": "per_school", "school_id": sid, "n_eval": len(g_test)})
        local_reports.append(loc_rep)

        # Pooled-per-school metrics (slice pooled preds for this school's test)
        mask_sid = (test_df["school_id"] == sid).values
        pooled_preds_sid = {q: pooled_preds_all[q][mask_sid] for q in QUANTILES}
        pool_rep = summarise_prob(y_te, pooled_preds_sid)
        pool_rep.update({"method": "pooled_gb_quantile", "level": "per_school", "school_id": sid, "n_eval": len(g_test)})
        pooled_reports.append(pool_rep)

        # Per-school baselines
        for name, lag in [("naive_day", LAG_DAY), ("naive_week", LAG_WEEK)]:
            yhat, mask = seasonal_naive_with_mask(g_train, g_test, lag_steps=lag)
            y_true = y_te[mask]
            yhat = yhat[mask]
            preds = degenerate_quantiles_from_point(yhat)
            brep = summarise_prob(y_true, preds)
            brep.update({"method": name, "level": "per_school", "school_id": sid, "n_eval": int(mask.sum())})
            per_school_rows.append(brep)

        preds_hs = historical_quantile_baseline(g_train, g_test)
        brep = summarise_prob(y_te, preds_hs)
        brep.update({"method": "hist_quantile_slot", "level": "per_school", "school_id": sid, "n_eval": len(g_test)})
        per_school_rows.append(brep)

        # Progress line
        elapsed = time.time() - t0
        avg = elapsed / k
        eta = avg * (n_schools - k)
        print(
            f"[WP1-FIX] {k:02d}/{n_schools} {sid}: {fmt_seconds(time.time()-t_school)} | "
            f"elapsed {fmt_seconds(elapsed)} | ETA {fmt_seconds(eta)}",
            flush=True
        )

    per_school_rows.extend(local_reports)
    per_school_rows.extend(pooled_reports)

    # ---- Bootstrap pooled vs local ----
    per_school_df = pd.DataFrame(per_school_rows)
    loc = per_school_df[per_school_df["method"] == "per_school_gb_quantile"].set_index("school_id")
    pool = per_school_df[(per_school_df["method"] == "pooled_gb_quantile") & (per_school_df["level"] == "per_school")].set_index("school_id")

    common = sorted(set(loc.index).intersection(set(pool.index)))
    loc = loc.loc[common]
    pool = pool.loc[common]

    dif_mae = (pool["mae_p50"] - loc["mae_p50"]).values
    dif_pin = (pool["pinball_mean"] - loc["pinball_mean"]).values

    mae_mean, mae_lo, mae_hi = paired_bootstrap(dif_mae)
    pin_mean, pin_lo, pin_hi = paired_bootstrap(dif_pin)

    win_mae = float(np.mean(dif_mae < 0))
    win_pin = float(np.mean(dif_pin < 0))

    boot_df = pd.DataFrame([{
        "metric": "mae_p50",
        "diff_mean_pooled_minus_local": mae_mean,
        "ci95_low": mae_lo,
        "ci95_high": mae_hi,
        "win_fraction_pooled_better": win_mae,
        "n_schools": len(common),
        "note": "negative diff => pooled better",
    }, {
        "metric": "pinball_mean",
        "diff_mean_pooled_minus_local": pin_mean,
        "ci95_low": pin_lo,
        "ci95_high": pin_hi,
        "win_fraction_pooled_better": win_pin,
        "n_schools": len(common),
        "note": "negative diff => pooled better",
    }])

    # ---- Save outputs ----
    overall_df = pd.DataFrame(overall_rows)
    overall_df.to_csv(OUT_OVERALL, index=False)
    per_school_df.to_csv(OUT_PER_SCHOOL, index=False)
    boot_df.to_csv(OUT_BOOT, index=False)

    print("\n[WP1-FIX] Wrote:", flush=True)
    print(f" - {OUT_OVERALL}", flush=True)
    print(f" - {OUT_PER_SCHOOL}", flush=True)
    print(f" - {OUT_BOOT}", flush=True)
    print("\n[WP1-FIX] Bootstrap pooled vs local:", flush=True)
    print(boot_df.to_string(index=False), flush=True)


if __name__ == "__main__":
    main()

[WP1-FIX] Loading: data/schools_panel.csv
[WP1-FIX] Rows train: 728,591 | test: 182,161
[WP1-FIX] Training pooled model (7 quantiles)...
[WP1-FIX] Pooled done in 45m48s
[WP1-FIX] Training per-school models (53 schools × 7 quantiles)...
[WP1-FIX] 01/53 S1: 38s | elapsed 38s | ETA 33m44s
[WP1-FIX] 02/53 S10: 36s | elapsed 1m15s | ETA 32m04s
[WP1-FIX] 03/53 S11: 39s | elapsed 1m54s | ETA 31m56s
[WP1-FIX] 04/53 S12: 38s | elapsed 2m33s | ETA 31m21s
[WP1-FIX] 05/53 S13: 39s | elapsed 3m12s | ETA 30m49s
[WP1-FIX] 06/53 S14: 39s | elapsed 3m52s | ETA 30m21s
[WP1-FIX] 07/53 S15: 37s | elapsed 4m29s | ETA 29m32s
[WP1-FIX] 08/53 S16: 38s | elapsed 5m07s | ETA 28m52s
[WP1-FIX] 09/53 S17: 39s | elapsed 5m47s | ETA 28m20s
[WP1-FIX] 10/53 S18: 38s | elapsed 6m26s | ETA 27m42s
[WP1-FIX] 11/53 S19: 38s | elapsed 7m05s | ETA 27m03s
[WP1-FIX] 12/53 S2: 40s | elapsed 7m45s | ETA 26m31s
[WP1-FIX] 13/53 S20: 37s | elapsed 8m23s | ETA 25m48s
[WP1-FIX] 14/53 S21: 38s | elapsed 9m01s | ETA 25m07s
[WP1-FIX] 15